# Chapter 29 — Embedding Matplotlib charts in Tkinter

**Book role:** Visualization  
**Suggested book space:** about 10 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Create charts from prepared DataFrame summaries.
- Embed a Matplotlib FigureCanvasTkAgg inside a Tk widget tree.
- Manage redraws and figure lifetimes.
- Select chart types and labels for data-analysis questions.

## 1. Prepare chart data before rendering

A chart should show a clear relationship: comparison across categories, trend over time, distribution, composition, or relationship between numeric variables. Select the right summary table before calling Matplotlib. Sort date series chronologically; sort categorical bars according to a meaningful order; label the measure and units; and handle empty or all-missing data deliberately.

Do not embed a large transformation pipeline inside a draw callback. Compute a chart-ready DataFrame in a service and pass it to a rendering function. Then the same aggregation can power a chart, an export, and a summary KPI. Data selection, analytical calculation, and presentation should remain distinct.

### Worked example 1: Embed a Matplotlib figure

This creates a Figure and embeds its canvas in a frame. Install Matplotlib in the same environment as Tkinter. For a production view, keep the canvas and figure on the view object and update it from prepared summaries.

In [ ]:
import tkinter as tk
from tkinter import ttk
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

root = tk.Tk()
root.title("Embedded chart")
root.geometry("680x420")
frame = ttk.Frame(root, padding=8)
frame.pack(fill="both", expand=True)
figure = Figure(figsize=(6, 3.5), dpi=100, constrained_layout=True)
ax = figure.add_subplot(111)
regions = ["North", "South", "West"]
revenue = [125000, 98000, 143000]
ax.bar(regions, revenue)
ax.set_title("Revenue by region")
ax.set_ylabel("Revenue (currency units)")
ax.grid(axis="y", alpha=0.25)
canvas = FigureCanvasTkAgg(figure, master=frame)
canvas.get_tk_widget().pack(fill="both", expand=True)
canvas.draw()
root.mainloop()

## 2. FigureCanvasTkAgg and widget ownership

Matplotlib’s Tk backend provides `FigureCanvasTkAgg` to embed a Figure into a Tk widget. Create a `Figure` directly (for example, `Figure(figsize=(6, 4), dpi=100)`) and create axes from it rather than calling `pyplot.show()` inside a Tkinter application. Pack or grid the canvas in a frame that can expand. `draw_idle()` can defer rendering until the event loop is ready, reducing redundant immediate redraws.

Keep references to the figure, axes, and canvas on the view object if you need to update the chart. When refreshing, clear the axes or reuse artists consistently; creating a new canvas every time leaks widgets and can make the interface slower. Update chart titles and labels with the actual filter range so users can see which data are represented.

### Worked example 2: Update an existing figure instead of creating duplicates

This reusable renderer expects prepared data and a reusable Axes. The calling view remains responsible for the canvas and scheduling the redraw on the Tk thread.

In [ ]:
def render_region_chart(ax, summary):
    """Redraw an existing Axes from a DataFrame with region and revenue columns."""
    ax.clear()
    if summary.empty:
        ax.text(0.5, 0.5, "No data for the current filters", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return
    ax.set_axis_on()
    ordered = summary.sort_values("revenue", ascending=False)
    ax.bar(ordered["region"], ordered["revenue"])
    ax.set_title("Revenue by region")
    ax.set_ylabel("Revenue")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25)

# In the view: render_region_chart(self.ax, summary); self.canvas.draw_idle()

## 3. Chart choice and statistical honesty

A bar chart is effective for category comparison; a line chart is appropriate for ordered time or continuous sequences; a histogram shows distributions; a scatter plot shows relationships; and a box plot summarizes spread and outliers under specific assumptions. Pie charts become hard to compare with many categories. Dual axes can mislead when scales are unrelated. Truncated axes may amplify visual differences in ways users do not expect.

Missing data, outliers, duplicate records, and status filters affect interpretation. Add sample size, date range, units, and source/aggregation definition where relevant. A chart should not hide uncertainty or imply causal relationships from descriptive associations. For student work, explain why the chart type answers the stated question rather than choosing a chart based solely on appearance.

## Guided lab: Lab: linked table and chart

Build a chart panel that displays sales by month and region. It should use the currently filtered data, display no-data states, show clear units and date range, reuse the existing canvas on refresh, and not block the UI while a slow query runs. Add a toggle for line versus bar chart only where both are defensible for the same analytical question.

## Review questions and practice

1. Why should aggregation be computed outside the chart renderer?
2. What does `draw_idle()` help with?
3. Why is `pyplot.show()` not the preferred embedding pattern?
4. Which chart would you choose for a month-over-month trend and why?
5. What context should a chart communicate besides its title?

## Further reading

- Matplotlib embedding in Tk: https://matplotlib.org/stable/gallery/user_interfaces/embedding_in_tk_sgskip.html
- Matplotlib figure API: https://matplotlib.org/stable/api/figure_api.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.